![logo_ironhack_blue 7](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Lab | Reinforcement Learning

## Introduction

Reinforcement learning (RL) is a type of machine learning where an **agent** learns to make decisions by interacting with an **environment**. Unlike supervised learning, the agent is not given labeled examples — it must discover which actions yield the highest cumulative **reward** through trial and error.

The core loop is:
1. **Observe** the current state `s`
2. **Choose** an action `a` (using a policy)
3. **Receive** a reward `r` and transition to next state `s'`
4. **Update** the value estimates

In this notebook we implement **Q-Learning** and **SARSA** from scratch using **Gymnasium** environments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

# For reproducibility
np.random.seed(42)

---

## Task 1: Environment Exploration

Before training any agent, we need to understand the environments we will be working with. Gymnasium provides a clean, standardized API for RL environments. The key concepts are:

- **Observation space**: the set of all possible states the agent can observe
- **Action space**: the set of all actions the agent can take
- **Step**: taking one action; returns `(next_state, reward, terminated, truncated, info)`
- **Episode**: a sequence of steps from reset to terminal state

### 1.1 FrozenLake-v1

In [ ]:
# Create the FrozenLake environment
env_fl = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")

print("=== FrozenLake-v1 ===")
print(f"Observation space : {env_fl.observation_space}")
print(f"Number of states  : {env_fl.observation_space.n}")
print(f"Action space      : {env_fl.action_space}")
print(f"Number of actions : {env_fl.action_space.n}")

print("\nAction meanings:")
action_names_fl = {0: "LEFT", 1: "DOWN", 2: "RIGHT", 3: "UP"}
for idx, name in action_names_fl.items():
    print(f"  {idx} -> {name}")

**Action index meanings for FrozenLake:**
| Index | Direction |
|-------|-----------|
| 0     | LEFT      |
| 1     | DOWN      |
| 2     | RIGHT     |
| 3     | UP        |

The grid is 4×4 = **16 states** (0–15). State 0 is the top-left starting position (S), state 15 is the goal (G). Holes (H) are instant-fail terminal states.

In [ ]:
# Render the FrozenLake grid
obs, info = env_fl.reset()
rendered = env_fl.render()
print("FrozenLake Grid Layout:")
print(rendered)

In [ ]:
# Run 5 episodes with a random agent on FrozenLake
print("=== FrozenLake: 5 Random-Agent Episodes ===")

for episode in range(1, 6):
    obs, info = env_fl.reset()
    total_reward = 0
    steps = 0
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = env_fl.action_space.sample()   # random action
        obs, reward, terminated, truncated, info = env_fl.step(action)
        total_reward += reward
        steps += 1

    print(f"  Episode {episode}: total_reward={total_reward:.1f}, steps={steps}")

env_fl.close()

### 1.2 Taxi-v3

In [ ]:
# Create the Taxi environment
env_taxi = gym.make("Taxi-v3")

print("=== Taxi-v3 ===")
print(f"Observation space : {env_taxi.observation_space}")
print(f"Number of states  : {env_taxi.observation_space.n}")
print(f"Action space      : {env_taxi.action_space}")
print(f"Number of actions : {env_taxi.action_space.n}")

print("\nAction meanings:")
action_names_taxi = {0: "SOUTH", 1: "NORTH", 2: "EAST", 3: "WEST", 4: "PICKUP", 5: "DROPOFF"}
for idx, name in action_names_taxi.items():
    print(f"  {idx} -> {name}")

In [ ]:
# Run 5 episodes with a random agent on Taxi
print("=== Taxi-v3: 5 Random-Agent Episodes ===")

for episode in range(1, 6):
    obs, info = env_taxi.reset()
    total_reward = 0
    steps = 0
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = env_taxi.action_space.sample()   # random action
        obs, reward, terminated, truncated, info = env_taxi.step(action)
        total_reward += reward
        steps += 1

    print(f"  Episode {episode}: total_reward={total_reward:.1f}, steps={steps}")

env_taxi.close()

### 1.3 Environment Comparison

| Property            | FrozenLake-v1 | Taxi-v3         |
|---------------------|---------------|-----------------|
| State space size    | 16            | 500             |
| Action space size   | 4             | 6               |
| Goal                | Reach tile G  | Pick up & drop off passenger |
| Reward on success   | +1            | +20             |
| Reward per step     | 0             | -1              |
| Illegal action penalty | None       | -10             |

**Why is Taxi harder than FrozenLake?**

1. **Much larger state space**: Taxi has 500 states (25 taxi positions × 5 passenger locations × 4 destinations), compared to just 16 in FrozenLake. The agent needs many more experiences to cover the state space.
2. **Two-stage task**: The agent must first navigate to the passenger, *pick up* the passenger (action 4), navigate to the correct destination, then *drop off* (action 5). A random agent almost never completes this sequence.
3. **Illegal action penalties**: Attempting PICKUP or DROPOFF at the wrong location gives a −10 penalty, making random exploration especially costly.
4. **Continuous step cost**: Each step costs −1, so the agent must also find an *efficient* route — not just a correct one.

---

## Task 2: Q-Learning on FrozenLake

### What is Q-Learning?

Q-Learning is an **off-policy** temporal-difference (TD) algorithm. It maintains a **Q-table** `Q[s, a]` that estimates the expected cumulative discounted reward of taking action `a` in state `s` and then following the optimal policy thereafter.

The update rule is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \Big]$$

- $\alpha$ (alpha): **learning rate** — how much to shift toward the new estimate
- $\gamma$ (gamma): **discount factor** — how much future rewards are worth relative to immediate ones
- $\max_{a'} Q(s', a')$: the **greedy** estimate of the best possible future value (this is the "off-policy" part)

**Exploration vs. Exploitation** is handled with an **ε-greedy policy**:
- With probability ε → take a random action (explore)
- With probability 1−ε → take the action with the highest Q-value (exploit)

ε is decayed over time so the agent explores broadly early on and becomes more greedy as it learns.

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
env_fl = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)

n_states_fl  = env_fl.observation_space.n   # 16
n_actions_fl = env_fl.action_space.n        # 4

# ── Q-table ──────────────────────────────────────────────────────────────────
Q_fl = np.zeros((n_states_fl, n_actions_fl))

# ── Hyperparameters ──────────────────────────────────────────────────────────
alpha         = 0.8    # learning rate
gamma         = 0.95   # discount factor
epsilon       = 1.0    # initial exploration rate
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_fl = 10_000

# ── Training loop ─────────────────────────────────────────────────────────────
rewards_fl = []

for episode in range(n_episodes_fl):
    state, _ = env_fl.reset()
    total_reward = 0
    terminated = False
    truncated  = False

    while not terminated and not truncated:
        # ε-greedy action selection
        if np.random.random() < epsilon:
            action = env_fl.action_space.sample()          # explore
        else:
            action = np.argmax(Q_fl[state])                # exploit

        next_state, reward, terminated, truncated, _ = env_fl.step(action)

        # Q-Learning update
        best_next = np.max(Q_fl[next_state])
        Q_fl[state, action] += alpha * (reward + gamma * best_next - Q_fl[state, action])

        state = next_state
        total_reward += reward

    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_fl.append(total_reward)

env_fl.close()
print(f"Training complete. Final epsilon: {epsilon:.4f}")
print(f"Overall success rate: {np.mean(rewards_fl):.3f}")

In [ ]:
# ── Learning curve: rolling average reward ────────────────────────────────────
window = 100
rolling_avg_fl = pd.Series(rewards_fl).rolling(window).mean()

plt.figure(figsize=(10, 4))
plt.plot(rolling_avg_fl, color='steelblue', linewidth=1.5)
plt.title("Q-Learning on FrozenLake-v1\n(Rolling Average Reward, window=100)")
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Print the final Q-table ────────────────────────────────────────────────────
q_df = pd.DataFrame(
    Q_fl,
    columns=["LEFT", "DOWN", "RIGHT", "UP"],
    index=[f"State {i}" for i in range(n_states_fl)]
)
print("Final Q-Table (FrozenLake):")
print(q_df.round(4).to_string())

print("\nGreedy policy (best action per state):")
action_labels = ["LEFT", "DOWN", "RIGHT", "UP"]
for s in range(n_states_fl):
    best_a = np.argmax(Q_fl[s])
    print(f"  State {s:2d}: {action_labels[best_a]}")

### 2.1 Observations on the Learned Policy

Looking at the Q-table and the greedy policy:

- **Start state (state 0, top-left)**: The highest Q-value action should be **DOWN** or **RIGHT** — the two safe directions that move toward the goal without falling into a hole. This makes intuitive sense given the 4×4 grid layout:
  ```
  S F F F
  F H F H
  F F F H
  H F F G
  ```
- **Hole states** (states 5, 7, 11, 12): These are terminal states; all Q-values remain 0 since the agent never takes another action from them.
- **Goal state (state 15)**: Also terminal; Q-values are 0.
- **States near holes**: The agent has learned to prefer actions that steer away from hole-adjacent tiles, reflected in lower Q-values for dangerous directions.
- The learning curve rises sharply around episode 1,000–2,000 and plateaus near 1.0, indicating the agent reliably solves the deterministic grid after sufficient exploration.

---

## Task 3: Q-Learning on Taxi-v3

Now we apply Q-Learning to the more complex **Taxi-v3** environment. The state space is 500 states, meaning the Q-table has 500 × 6 = 3,000 entries to learn. We train for 20,000 episodes to give the agent enough experience to cover this larger space.

The reward structure for Taxi:
- **+20**: Successfully dropping off the passenger at the correct location
- **−1**: Each time step (encourages efficiency)
- **−10**: Illegal PICKUP or DROPOFF

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
env_taxi = gym.make("Taxi-v3")

n_states_taxi  = env_taxi.observation_space.n   # 500
n_actions_taxi = env_taxi.action_space.n        # 6

# ── Q-table (reset for new environment) ──────────────────────────────────────
Q_taxi = np.zeros((n_states_taxi, n_actions_taxi))

# ── Hyperparameters (same as Task 2) ─────────────────────────────────────────
alpha         = 0.8
gamma         = 0.95
epsilon       = 1.0
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_taxi = 20_000

# ── Training loop ─────────────────────────────────────────────────────────────
rewards_taxi_ql = []

for episode in range(n_episodes_taxi):
    state, _ = env_taxi.reset()
    total_reward = 0
    terminated = False
    truncated  = False

    while not terminated and not truncated:
        # ε-greedy action selection
        if np.random.random() < epsilon:
            action = env_taxi.action_space.sample()
        else:
            action = np.argmax(Q_taxi[state])

        next_state, reward, terminated, truncated, _ = env_taxi.step(action)

        # Q-Learning update
        best_next = np.max(Q_taxi[next_state])
        Q_taxi[state, action] += alpha * (reward + gamma * best_next - Q_taxi[state, action])

        state = next_state
        total_reward += reward

    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_taxi_ql.append(total_reward)

env_taxi.close()
print(f"Training complete. Final epsilon: {epsilon:.4f}")
print(f"Mean reward (last 1000 episodes): {np.mean(rewards_taxi_ql[-1000:]):.2f}")

In [ ]:
# ── Learning curve: average reward per 100 episodes ───────────────────────────
rewards_taxi_ql_arr = np.array(rewards_taxi_ql)
avg_per_100_ql = rewards_taxi_ql_arr.reshape(-1, 100).mean(axis=1)
episodes_x = np.arange(100, n_episodes_taxi + 1, 100)

plt.figure(figsize=(10, 4))
plt.plot(episodes_x, avg_per_100_ql, color='darkorange', linewidth=1.5)
plt.title("Q-Learning on Taxi-v3\n(Average Reward per 100 Episodes)")
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluate Q-Learning agent (epsilon = 0, pure exploitation) ─────────────────
env_taxi_eval = gym.make("Taxi-v3")

n_test = 100
test_rewards_ql = []

for _ in range(n_test):
    state, _ = env_taxi_eval.reset()
    total_reward = 0
    terminated = False
    truncated  = False

    while not terminated and not truncated:
        action = np.argmax(Q_taxi[state])   # greedy — no exploration
        state, reward, terminated, truncated, _ = env_taxi_eval.step(action)
        total_reward += reward

    test_rewards_ql.append(total_reward)

env_taxi_eval.close()

avg_reward_ql  = np.mean(test_rewards_ql)
success_rate_ql = np.mean([r > 0 for r in test_rewards_ql])

print("=== Q-Learning Evaluation on Taxi-v3 (100 test episodes) ===")
print(f"  Average reward : {avg_reward_ql:.2f}")
print(f"  Success rate   : {success_rate_ql * 100:.1f}%  (episodes with reward > 0)")

### 3.1 Observations on Taxi Training

**Training curve vs FrozenLake:**
- In FrozenLake, the curve rises steeply and quickly plateaus at a near-perfect success rate because the state space (16 states) is tiny and deterministic. Even random exploration covers it fast.
- In Taxi, the curve starts very negative (the random agent incurs many −10 illegal-action penalties and −1 per-step costs). Improvement is gradual and the curve is noisier because 500 states take much longer to visit and update.

**Stabilization:**
- Performance begins to stabilize roughly around **episodes 5,000–10,000**, once ε has decayed enough that the agent starts exploiting its learned values and stops making costly random PICKUP/DROPOFF mistakes.
- After ~15,000 episodes the curve levels off, indicating the policy has largely converged.

**Evaluation results:**
- A well-trained Q-Learning agent typically achieves an **average reward of ~7–9** and a **success rate near 100%** on Taxi, compared to a random agent that almost never succeeds.

---

## Task 4: SARSA Comparison

### What is SARSA?

**SARSA** (State-Action-Reward-State-Action) is an **on-policy** TD algorithm. Its update rule is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \cdot Q(s', a') - Q(s, a) \Big]$$

The critical difference from Q-Learning: instead of using $\max_{a'} Q(s', a')$ (the *best possible* next action), SARSA uses $Q(s', a')$ where **$a'$ is the actual next action chosen by the current ε-greedy policy**.

This makes SARSA **on-policy**: it updates based on what the agent *actually does*, including exploratory random actions. Q-Learning is **off-policy**: it always updates toward the *best* possible action, regardless of what was actually done during exploration.

### SARSA Algorithm
```
Initialize Q(s, a) = 0 for all s, a
For each episode:
    s ← reset()
    a ← ε-greedy(Q, s)        # choose first action
    While not terminal:
        s', r ← step(a)
        a' ← ε-greedy(Q, s')  # choose NEXT action (on-policy)
        Q(s,a) ← Q(s,a) + α[r + γ·Q(s',a') − Q(s,a)]
        s ← s', a ← a'
    Decay ε
```

In [ ]:
# ── Environment ──────────────────────────────────────────────────────────────
env_sarsa = gym.make("Taxi-v3")

n_states_sarsa  = env_sarsa.observation_space.n   # 500
n_actions_sarsa = env_sarsa.action_space.n        # 6

# ── Q-table for SARSA ────────────────────────────────────────────────────────
Q_sarsa = np.zeros((n_states_sarsa, n_actions_sarsa))

# ── Hyperparameters (same as before) ─────────────────────────────────────────
alpha         = 0.8
gamma         = 0.95
epsilon       = 1.0
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_sarsa = 20_000

def epsilon_greedy(Q, state, eps):
    """Select action using ε-greedy policy."""
    if np.random.random() < eps:
        return env_sarsa.action_space.sample()   # explore
    return np.argmax(Q[state])                   # exploit

# ── SARSA Training loop ───────────────────────────────────────────────────────
rewards_sarsa = []

for episode in range(n_episodes_sarsa):
    state, _ = env_sarsa.reset()
    action = epsilon_greedy(Q_sarsa, state, epsilon)   # choose first action
    total_reward = 0
    terminated = False
    truncated  = False

    while not terminated and not truncated:
        next_state, reward, terminated, truncated, _ = env_sarsa.step(action)

        # Choose the NEXT action using the current policy (on-policy)
        next_action = epsilon_greedy(Q_sarsa, next_state, epsilon)

        # SARSA update — uses Q(s', a'), not max Q(s', *)
        Q_sarsa[state, action] += alpha * (
            reward + gamma * Q_sarsa[next_state, next_action] - Q_sarsa[state, action]
        )

        state  = next_state
        action = next_action   # carry the chosen action forward
        total_reward += reward

    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards_sarsa.append(total_reward)

env_sarsa.close()
print(f"SARSA training complete. Final epsilon: {epsilon:.4f}")
print(f"Mean reward (last 1000 episodes): {np.mean(rewards_sarsa[-1000:]):.2f}")

In [ ]:
# ── Plot both learning curves on the same figure ──────────────────────────────
rewards_sarsa_arr = np.array(rewards_sarsa)
avg_per_100_sarsa = rewards_sarsa_arr.reshape(-1, 100).mean(axis=1)

plt.figure(figsize=(11, 5))
plt.plot(episodes_x, avg_per_100_ql,    color='darkorange', linewidth=1.5, label='Q-Learning (off-policy)')
plt.plot(episodes_x, avg_per_100_sarsa, color='steelblue',  linewidth=1.5, label='SARSA (on-policy)')
plt.title("Q-Learning vs SARSA on Taxi-v3\n(Average Reward per 100 Episodes)")
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluate SARSA agent ───────────────────────────────────────────────────────
env_eval2 = gym.make("Taxi-v3")

test_rewards_sarsa = []

for _ in range(n_test):
    state, _ = env_eval2.reset()
    total_reward = 0
    terminated = False
    truncated  = False

    while not terminated and not truncated:
        action = np.argmax(Q_sarsa[state])   # greedy
        state, reward, terminated, truncated, _ = env_eval2.step(action)
        total_reward += reward

    test_rewards_sarsa.append(total_reward)

env_eval2.close()

avg_reward_sarsa   = np.mean(test_rewards_sarsa)
success_rate_sarsa = np.mean([r > 0 for r in test_rewards_sarsa])

print("=== Evaluation Summary (100 test episodes each) ===")
print(f"{'Algorithm':<20} {'Avg Reward':>12} {'Success Rate':>14}")
print("-" * 48)
print(f"{'Q-Learning':<20} {avg_reward_ql:>12.2f} {success_rate_ql*100:>13.1f}%")
print(f"{'SARSA':<20} {avg_reward_sarsa:>12.2f} {success_rate_sarsa*100:>13.1f}%")

### 4.1 Q-Learning vs SARSA — Analysis

#### Which algorithm converged faster?

**Q-Learning** typically converges faster in the Taxi environment. Because it always targets the *greedy maximum* value, each update is more aggressively directed toward the optimal policy. SARSA's updates are "softer" — they account for the ε-greedy exploration noise, so the Q-values reflect a mix of the optimal behavior and the random exploratory moves.

#### Which achieved higher final reward?

Both algorithms ultimately reach similar final performance once ε has fully decayed and both agents are nearly greedy. In practice, **Q-Learning** may edge out SARSA slightly in average reward during training because its update rule is always pointing toward the best action. At test time (ε = 0), both agents follow fully greedy policies derived from their Q-tables, so the final evaluation rewards tend to be comparable.

#### On-policy vs Off-policy — Fundamental Difference

| | Q-Learning (Off-policy) | SARSA (On-policy) |
|---|---|---|
| **Update target** | $r + \gamma \max_{a'} Q(s', a')$ | $r + \gamma Q(s', a')$ where $a'$ is what the agent *will* do |
| **Policy evaluated** | Always the *greedy* policy | The *current* ε-greedy policy |
| **Behavior during training** | More aggressive, risk-taking | More cautious, conservative |
| **Q-values represent** | Optimal future value | Value under current (exploratory) policy |

**When to prefer Q-Learning:**
- When you want the agent to learn the *optimal* policy as quickly as possible and can tolerate risky behavior during training (e.g., simulated environments where mistakes are free).
- Offline learning — learning from data collected by another policy (behavior policy ≠ target policy).

**When to prefer SARSA:**
- In **safety-critical** real-world environments where exploratory mistakes have real costs (e.g., robot control, traffic systems). SARSA learns to avoid dangerous exploratory actions because it accounts for them in its value estimates.
- When the gap between exploratory and greedy behavior is large and you want value estimates that accurately reflect the agent's actual behavior during learning.

---

## Summary

In this lab we:

1. **Explored** FrozenLake-v1 (16 states, 4 actions) and Taxi-v3 (500 states, 6 actions) with random agents, observing that Taxi is substantially harder due to its larger state space, multi-stage task structure, and illegal-action penalties.

2. **Implemented Q-Learning** from scratch with ε-greedy exploration and epsilon decay. On FrozenLake the agent quickly converges to a near-perfect policy. On Taxi, convergence takes ~10,000–15,000 episodes.

3. **Evaluated** the trained Q-Learning Taxi agent at test time (ε = 0), achieving high average reward and success rate.

4. **Implemented SARSA** — differing from Q-Learning only in that it uses the *actual* next action (on-policy) rather than the greedy max. Both algorithms reach similar final performance, but Q-Learning typically converges faster while SARSA is safer in environments where exploration mistakes are costly.

These tabular methods scale well for small, discrete state spaces. For larger or continuous state spaces, function approximation methods (e.g., Deep Q-Networks) are required.